# Chapter 1 Preprocessing Runbook

This notebook is a thin orchestration and QC layer for the canonical Chapter 1 preprocessing pipeline in this repository.

- It reads standardized upstream ASIC artifacts from explicit artifact paths.
- It uses the shared JSON run config that is also used by the CLI.
- It calls the package pipeline once, then displays the cohort, instance, label, and model-ready outputs stage by stage.
- It writes final outputs into this repo's artifact directory.


## Configuration

Edit the shared run config file before executing the notebook end to end. The default repo config points at the standardized ASIC artifact root in `icu-data-platform`.


In [1]:
from pathlib import Path

RUN_CONFIG_PATH = Path("config") / "ch1_run_config.json"


In [2]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "chapter1_mortality_decomposition").exists():
    if (PROJECT_ROOT.parent / "src" / "chapter1_mortality_decomposition").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError("Run this notebook from the repo root or the notebooks directory.")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import chapter1_mortality_decomposition.artifacts as artifacts_module
import chapter1_mortality_decomposition.pipeline as pipeline_module
import chapter1_mortality_decomposition.run_config as run_config_module

artifacts_module = importlib.reload(artifacts_module)
pipeline_module = importlib.reload(pipeline_module)
run_config_module = importlib.reload(run_config_module)

load_chapter1_inputs = artifacts_module.load_chapter1_inputs
build_chapter1_dataset = pipeline_module.build_chapter1_dataset
write_chapter1_dataset = pipeline_module.write_chapter1_dataset
load_chapter1_run_config = run_config_module.load_chapter1_run_config

pd.set_option("display.max_colwidth", 160)

run_config = load_chapter1_run_config(PROJECT_ROOT / RUN_CONFIG_PATH)
if not hasattr(run_config, "split_random_seed"):
    raise AttributeError(
        "Loaded Chapter1RunConfig does not expose split_random_seed. Restart the kernel and rerun the notebook from the top."
    )

config = run_config.to_chapter1_config()
input_dir = run_config.input_dir
output_dir = run_config.output_dir

display(Markdown(f"**Project root:** `{PROJECT_ROOT}`"))
display(Markdown(f"**Run config:** `{run_config.run_config_path}`"))
display(Markdown(f"**Using run_config module:** `{Path(run_config_module.__file__).resolve()}`"))
display(
    pd.DataFrame(
        [
            {"setting": "input_dir", "value": str(input_dir)},
            {"setting": "input_format", "value": run_config.input_format},
            {"setting": "output_dir", "value": str(output_dir)},
            {"setting": "output_format", "value": run_config.output_format},
            {"setting": "feature_set_config_path", "value": str(run_config.feature_set_config_path)},
            {"setting": "horizons_hours", "value": list(run_config.horizons_hours)},
            {"setting": "min_required_core_groups", "value": run_config.min_required_core_groups},
            {"setting": "split_random_seed", "value": run_config.split_random_seed},
            {
                "setting": "mech_vent_stay_qc_artifact",
                "value": str(input_dir / "qc" / f"mech_vent_ge_24h_stay_level.{run_config.input_format}"),
            },
            {
                "setting": "mech_vent_episode_qc_artifact",
                "value": str(input_dir / "qc" / f"mech_vent_ge_24h_episode_level.{run_config.input_format}"),
            },
        ]
    )
)


**Project root:** `/Users/joanameyer/repository/1-mortality-decomposition`

**Run config:** `/Users/joanameyer/repository/1-mortality-decomposition/config/ch1_run_config.json`

**Using run_config module:** `/Users/joanameyer/repository/1-mortality-decomposition/src/chapter1_mortality_decomposition/run_config.py`

,setting,value
0,input_dir,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized
1,input_format,csv
2,output_dir,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1
3,output_format,csv
4,feature_set_config_path,/Users/joanameyer/repository/1-mortality-decomposition/config/ch1_feature_sets.json
5,horizons_hours,"[8, 16, 24, 48, 72]"
6,min_required_core_groups,3
7,split_random_seed,20260327
8,mech_vent_stay_qc_artifact,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/qc/mech_vent_ge_24h_stay_level.csv
9,mech_vent_episode_qc_artifact,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/qc/mech_vent_ge_24h_episode_level.csv


In [3]:
if not input_dir.exists():
    raise FileNotFoundError(
        f"Input directory does not exist: {input_dir}. Update {run_config.run_config_path}."
    )

inputs = load_chapter1_inputs(input_dir=input_dir, input_format=run_config.input_format)

artifact_overview = pd.DataFrame(
    [
        {"table": "static_harmonized", "rows": inputs.static_harmonized.shape[0], "columns": inputs.static_harmonized.shape[1]},
        {"table": "dynamic_harmonized", "rows": inputs.dynamic_harmonized.shape[0], "columns": inputs.dynamic_harmonized.shape[1]},
        {"table": "block_index", "rows": inputs.block_index.shape[0], "columns": inputs.block_index.shape[1]},
        {"table": "blocked_dynamic_features", "rows": inputs.blocked_dynamic_features.shape[0], "columns": inputs.blocked_dynamic_features.shape[1]},
        {"table": "stay_block_counts", "rows": inputs.stay_block_counts.shape[0], "columns": inputs.stay_block_counts.shape[1]},
        {"table": "mech_vent_stay_level_qc", "rows": inputs.mech_vent_stay_level_qc.shape[0], "columns": inputs.mech_vent_stay_level_qc.shape[1]},
        {"table": "mech_vent_episode_level", "rows": inputs.mech_vent_episode_level.shape[0], "columns": inputs.mech_vent_episode_level.shape[1]},
    ]
)
display(Markdown("## Loaded Upstream Artifacts"))
display(artifact_overview)


## Loaded Upstream Artifacts

,table,rows,columns
0,static_harmonized,80,16
1,dynamic_harmonized,105165,107
2,block_index,3250,6
3,blocked_dynamic_features,3250,621
4,stay_block_counts,80,9
5,mech_vent_stay_level_qc,80,6
6,mech_vent_episode_level,202,7


In [4]:
dataset = build_chapter1_dataset(inputs, config=config)


In [5]:
display(Markdown("## Cohort QC"))
display(pd.DataFrame([
    {
        "retained_hospitals": dataset.cohort.retained_hospitals.shape[0],
        "retained_stays": dataset.cohort.table.shape[0],
    }
]))
display(Markdown("### Site-level eligibility"))
display(dataset.cohort.site_eligibility[["hospital_id", "site_included_ch1", "usable_core_vital_group_count", "exclusion_reason"]])
display(Markdown("### Stay-level exclusions"))
display(dataset.cohort.stay_exclusion_summary_by_hospital)
display(Markdown("### Canonical retained stay cohort"))
display(dataset.cohort.table)
display(Markdown("### Canonical cohort summary"))
display(dataset.cohort_summary)
display(Markdown("### Verification checks"))
display(dataset.verification_summary)


## Cohort QC

,retained_hospitals,retained_stays
0,4,35


### Site-level eligibility

,hospital_id,site_included_ch1,usable_core_vital_group_count,exclusion_reason
0,asic_UK00,False,4,missing/unusable icu_mortality at site level
1,asic_UK01,False,0,missing/unusable icu_mortality at site level; insufficient core-vitals coverage
2,asic_UK02,True,4,<NA>
3,asic_UK03,False,3,missing/unusable icu_mortality at site level
4,asic_UK04,True,4,<NA>
5,asic_UK06,False,0,insufficient core-vitals coverage
6,asic_UK07,True,4,<NA>
7,asic_UK08,True,4,<NA>


### Stay-level exclusions

...

,stay_id_global,hospital_id,readmission,icu_mortality,icd10_codes,icu_admission_time,icu_end_time_proxy,icu_end_time_proxy_hours,has_dynamic_data,mech_vent_ge_24h_qc,adult_age_ge_18_assumed_upstream,first_stay_proxy_eligible
0,asic_UK02_9990,asic_UK02,0.0,0.0,"Z93,E03,J96,I10,R77,C56,J90,E87,S31,U69,D68,Z43,I65,Z85,E11,Z92,D62,I69",0,12 days 22:30:00,310.50,True,True,True,True
1,asic_UK02_9991,asic_UK02,0.0,0.0,"R77,U69,Z85,N39,I10,C91,A49,J15,I48,T84,Z96,J96,B95,D68,D62,D69,R65,M00,A41,T84,Z96,B95",0,20 days 16:30:00,496.50,True,True,True,True
2,asic_UK02_9992,asic_UK02,0.0,1.0,"S27,E03,J96,M17,J98,U99,J93,I50,U69,G20,Z11,Z85,E11,I46",0,1 days 12:15:00,36.25,True,True,True,True
3,asic_UK02_9993,asic_UK02,0.0,0.0,"C09,E03,Z93,I10,E87,Z92,I25,Z43,C10",0,4 days 23:30:00,119.50,True,True,True,True
4,asic_UK02_9994,asic_UK02,0.0,1.0,"C34,J96,K86,J44,I10,I11,G40",0,35 days 09:15:00,849.25,True,True,True,True
5,asic_UK02_9995,asic_UK02,0.0,0.0,"R77,U69,N08,Z29,E11,R60,N17,I10,J80.03,E21,M31,Z11,I31,J96,D62,U07.1!,J12,D69,Z88,D65,I46",0,16 days 13:00:00,397.00,True,True,True,True
6,asic_UK02_9996,asic_UK02,0.0,0.0,"G93,I64,J18,E23,I60,U69,I61,I67",0,2 days 20:45:00,68.75,True,True,True,True
7,asic_UK02_9997,asic_UK02,0.0,1.0,"T81,R77,E86,E11,I25,E03,I10,I48,E06,J96,B95,E87,F01,I50,K92,M86,T89,D63,N18",0,11 days 19:45:00,283.75,True,True,True,True
8,asic_UK02_9998,asic_UK02,0.0,1.0,"I74,N17,Z96,I70,R57,R77,M62,E87,D69,C91,U69,R65,L89,U80,D68,E89,A41,Z22",0,11 days 16:15:00,280.25,True,True,True,True
9,asic_UK02_9999,asic_UK02,0.0,0.0,"T81,R00,R77,U69,Z29,L89,E11,Z43,B37,Z93,N17,E03,E78,I10,E05,E23,J80.03,A49,Z11,I31,J96,B96,D68,D62,U07.1!,J12,D69,Z22",0,7 days 19:15:00,187.25,True,True,True,True


### Canonical cohort summary

,summary_group,metric,horizon_h,value
0,cohort,input_hospitals,<NA>,8
1,cohort,retained_hospitals,<NA>,4
2,cohort,input_stays,<NA>,80
3,cohort,retained_stays,<NA>,35
4,instances,valid_prediction_instances_total,<NA>,8390
5,instances,valid_prediction_instances,8,1678
6,instances,valid_prediction_instances,16,1678
7,instances,valid_prediction_instances,24,1678
8,instances,valid_prediction_instances,48,1678
9,instances,valid_prediction_instances,72,1678


### Verification checks

,check_id,passed,detail
0,no_excluded_hospital_contributes_retained_stays,True,All retained stays and valid prediction instances come from site-eligible hospitals.
1,no_mech_vent_failed_stay_retained,True,No retained stay fails the upstream mech_vent_ge_24h_qc contract.
2,no_missing_or_readmission_flagged_stay_retained,True,No retained stay has missing readmission or readmission == 1.
3,valid_instances_restricted_to_retained_stays,True,Valid prediction instances are computed only on retained stays.
4,proxy_label_counts_consistent_with_valid_instances,True,"Per-horizon labelable and unlabeled counts sum to the valid-instance counts, and usable label rows match the total labelable count."


In [6]:
display(Markdown("## Feature-set Configuration and Validation"))
display(dataset.feature_set_validation_summary[[
    "feature_set_name",
    "primary_feature_count",
    "extended_only_feature_count",
    "total_extended_feature_count",
    "available_in_blocked_schema_count",
    "missing_from_blocked_schema_count",
    "absent_from_retained_data_count",
    "missing_from_blocked_schema_features",
    "absent_from_retained_data_features",
]])


## Feature-set Configuration and Validation

,feature_set_name,primary_feature_count,extended_only_feature_count,total_extended_feature_count,available_in_blocked_schema_count,missing_from_blocked_schema_count,absent_from_retained_data_count,missing_from_blocked_schema_features,absent_from_retained_data_features
0,primary,31,15,46,31,0,0,[],[]
1,extended,31,15,46,46,0,0,[],[]


In [7]:
display(Markdown("## Valid Prediction Instances"))
display(dataset.valid_instances.counts_by_horizon)
display(dataset.valid_instances.exclusion_summary)

display(Markdown("## Proxy Horizon Labels"))
display(dataset.labels.summary_by_horizon)
display(dataset.labels.unlabeled_reason_summary)


## Valid Prediction Instances

,horizon_h,candidate_instances,valid_instances,excluded_instances
0,8,1704,1678,26
1,16,1704,1678,26
2,24,1704,1678,26
3,48,1704,1678,26
4,72,1704,1678,26


,horizon_h,exclusion_reason,instance_count
0,8,insufficient_core_vital_group_coverage_in_block,26
1,16,insufficient_core_vital_group_coverage_in_block,26
2,24,insufficient_core_vital_group_coverage_in_block,26
3,48,insufficient_core_vital_group_coverage_in_block,26
4,72,insufficient_core_vital_group_coverage_in_block,26


## Proxy Horizon Labels

,horizon_h,total_valid_prediction_instances,labelable_instances,positive_labels,negative_labels,unlabeled_instances
0,8,1678,1256,10,1246,422
1,16,1678,1244,20,1224,434
2,24,1678,1232,30,1202,446
3,48,1678,1194,58,1136,484
4,72,1678,1154,82,1072,524


,horizon_h,unlabeled_reason,instance_count
0,8,missing_required_fields,0
1,8,prediction_time_not_strictly_before_proxy_end,1
2,8,survivor_without_full_horizon_observation,21
3,8,non_survivor_proxy_end_not_within_horizon,400
4,8,unsupported_icu_mortality_code,0
5,16,missing_required_fields,0
6,16,prediction_time_not_strictly_before_proxy_end,1
7,16,survivor_without_full_horizon_observation,43
8,16,non_survivor_proxy_end_not_within_horizon,390
9,16,unsupported_icu_mortality_code,0


In [8]:
display(Markdown("## Carry-forward and Missingness QC"))
for feature_set_name, feature_set_dataset in dataset.feature_sets.items():
    display(Markdown(f"### {feature_set_name.title()} Carry-forward Summary"))
    display(feature_set_dataset.model_ready.locf_feature_summary)
    display(feature_set_dataset.model_ready.ventilator_locf_summary)
    display(feature_set_dataset.model_ready.missingness_by_hospital_and_family)
    display(feature_set_dataset.model_ready.carry_forward_verification_summary)

display(Markdown("## Model-ready Datasets"))
final_qc_rows = []
for feature_set_name, feature_set_dataset in dataset.feature_sets.items():
    display(Markdown(f"### {feature_set_name.title()} Feature Set"))
    display(feature_set_dataset.model_ready.readiness_summary)
    display(feature_set_dataset.model_ready.split_summary)
    display(feature_set_dataset.model_ready.split_verification_summary)
    readiness = feature_set_dataset.model_ready.readiness_summary.set_index("metric")
    final_qc_rows.append(
        {
            "feature_set_name": feature_set_name,
            "model_ready_rows_total": int(feature_set_dataset.model_ready.table.shape[0]),
            "selected_feature_columns_total": readiness.at["selected_feature_columns_total", "value"],
            "configured_base_features_total": readiness.at["configured_base_features_total", "value"],
            "distinct_splits_total": readiness.at["distinct_splits_total", "value"],
            "global_median_imputation_applied_in_export": readiness.at["global_median_imputation_applied_in_export", "value"],
        }
    )
display(Markdown("### Primary vs Extended Summary"))
display(pd.DataFrame(final_qc_rows))


## Carry-forward and Missingness QC

### Primary Carry-forward Summary

,feature_set_name,qc_unit,base_variable,feature_family,locf_window_hours,ventilation_window_restricted,prediction_instances_total,originally_observed_instances,locf_filled_instances,remaining_missing_instances
0,primary,unique_prediction_instances_before_horizon_duplication,heart_rate,fast_bedside_physiology,4,False,1328,1328,0,0
1,primary,unique_prediction_instances_before_horizon_duplication,sbp,fast_bedside_physiology,4,False,1328,1127,0,201
2,primary,unique_prediction_instances_before_horizon_duplication,map,fast_bedside_physiology,4,False,1328,1127,0,201
3,primary,unique_prediction_instances_before_horizon_duplication,dbp,fast_bedside_physiology,4,False,1328,1127,0,201
4,primary,unique_prediction_instances_before_horizon_duplication,resp_rate,fast_bedside_physiology,4,False,1328,1259,0,69
5,primary,unique_prediction_instances_before_horizon_duplication,core_temp,fast_bedside_physiology,4,False,1328,998,0,330
6,primary,unique_prediction_instances_before_horizon_duplication,spo2,fast_bedside_physiology,4,False,1328,1325,0,3
7,primary,unique_prediction_instances_before_horizon_duplication,sao2,fast_bedside_physiology,4,False,1328,1079,0,249
8,primary,unique_prediction_instances_before_horizon_duplication,fio2,ventilator_variables,24,True,1328,1212,0,116
9,primary,unique_prediction_instances_before_horizon_duplication,peep,ventilator_variables,24,True,1328,1178,19,131


,feature_set_name,qc_unit,base_variable,locf_fills_inside_ventilation_window,locf_fills_outside_ventilation_window,no_locf_fills_outside_ventilation_window
0,primary,unique_prediction_instances_before_horizon_duplication,fio2,0,0,True
1,primary,unique_prediction_instances_before_horizon_duplication,peep,19,0,True
2,primary,unique_prediction_instances_before_horizon_duplication,vt,5,0,True
3,primary,unique_prediction_instances_before_horizon_duplication,vt_per_kg_ibw,18,0,True
4,primary,unique_prediction_instances_before_horizon_duplication,etco2,20,0,True


,feature_set_name,qc_unit,hospital_id,feature_family,base_feature_count,prediction_instances_total,feature_instance_cells_total,missing_before_locf_count,missing_after_locf_count,missing_before_locf_proportion,missing_after_locf_proportion
0,primary,unique_prediction_instances_before_horizon_duplication,asic_UK02,fast_bedside_physiology,8,219,1752,515,515,0.293950,0.293950
1,primary,unique_prediction_instances_before_horizon_duplication,asic_UK02,ventilator_variables,5,219,1095,377,376,0.344292,0.343379
2,primary,unique_prediction_instances_before_horizon_duplication,asic_UK02,no_bounded_locf_configured,1,219,219,41,41,0.187215,0.187215
3,primary,unique_prediction_instances_before_horizon_duplication,asic_UK02,blood_gas_acid_base,5,219,1095,271,259,0.247489,0.236530
4,primary,unique_prediction_instances_before_horizon_duplication,asic_UK02,lactate,1,219,219,13,13,0.059361,0.059361
5,primary,unique_prediction_instances_before_horizon_duplication,asic_UK02,standard_labs,7,219,1533,792,32,0.516634,0.020874
6,primary,unique_prediction_instances_before_horizon_duplication,asic_UK02,slower_labs,4,219,876,655,231,0.747717,0.263699
7,primary,unique_prediction_instances_before_horizon_duplication,asic_UK04,fast_bedside_physiology,8,217,1736,27,27,0.015553,0.015553
8,primary,unique_prediction_instances_before_horizon_duplication,asic_UK04,ventilator_variables,5,217,1085,446,442,0.411060,0.407373
9,primary,unique_prediction_instances_before_horizon_duplication,asic_UK04,no_bounded_locf_configured,1,217,217,61,61,0.281106,0.281106


...

,feature_set_name,metric,value,note
0,extended,model_ready_rows_total,6080,"Rows after valid-instance selection, proxy label availability filtering, bounded LOCF preprocessing, and split annotation."
1,extended,configured_base_features_total,46,Configured base features for this Chapter 1 feature set.
2,extended,selected_feature_columns_total,276,Blocked dynamic feature columns selected by the Chapter 1 feature config. obs_count columns remain raw; value-summary columns may be LOCF-filled.
3,extended,locf_missingness_indicator_columns_total,143,"Per-feature indicators appended to distinguish originally observed values, bounded-LOCF fills, and values still missing after LOCF."
4,extended,distinct_horizons_total,5,Configured prediction horizons represented in the model-ready table.
5,extended,distinct_splits_total,3,Stay-level train/validation/test splits represented in the model-ready table.
6,extended,bounded_locf_applied_in_export,True,Configured feature families use bounded preprocessing-time LOCF. Values remain missing when no valid carry-forward source exists.
7,extended,ventilator_locf_restricted_to_supported_windows,True,Ventilator-variable LOCF is restricted to blocks overlapping upstream ventilation-supported episodes.
8,extended,global_median_imputation_applied_in_export,False,No global median or other final imputation is applied in the preprocessing export.
9,extended,downstream_imputation_policy,deferred_to_model_training_stage,"Any final imputation must be fit later on the training split only, not during preprocessing export generation."


,feature_set_name,summary_level,split,hospital_id,horizon_h,stay_count,instance_count,positive_labels,negative_labels,label_prevalence
0,extended,overall,test,<NA>,<NA>,4,1111,0,1111,0.0
1,extended,overall_horizon,test,<NA>,8,4,235,0,235,0.0
2,extended,overall_horizon,test,<NA>,16,4,231,0,231,0.0
3,extended,overall_horizon,test,<NA>,24,4,227,0,227,0.0
4,extended,overall_horizon,test,<NA>,48,4,215,0,215,0.0
...,...,...,...,...,...,...,...,...,...,...
85,extended,hospital_horizon,validation,asic_UK08,8,1,155,0,155,0.0
86,extended,hospital_horizon,validation,asic_UK08,16,1,155,0,155,0.0
87,extended,hospital_horizon,validation,asic_UK08,24,1,155,0,155,0.0
88,extended,hospital_horizon,validation,asic_UK08,48,1,155,0,155,0.0


,feature_set_name,check_id,passed,detail
0,extended,no_unassigned_instances_in_model_ready,True,Every model-ready row inherits a split from the stay-level split table.
1,extended,all_instances_from_stay_share_same_split,True,All model-ready rows from the same stay_id_global share one split.


### Primary vs Extended Summary

,feature_set_name,model_ready_rows_total,selected_feature_columns_total,configured_base_features_total,distinct_splits_total,global_median_imputation_applied_in_export
0,primary,6080,186,31,3,False
1,extended,6080,276,46,3,False


In [9]:
output_paths = write_chapter1_dataset(
    dataset,
    output_dir=output_dir,
    output_format=run_config.output_format,
)

display(Markdown("## Outputs Written"))
output_manifest = pd.DataFrame(
    {
        "artifact_key": list(output_paths.keys()),
        "path": [str(path) for path in output_paths.values()],
    }
)
display(output_manifest)

print(f"Saved {len(output_paths)} output tables to {output_dir}")


## Outputs Written

,artifact_key,path
0,cohort_chapter1_notes,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_notes.csv
1,cohort_chapter1_core_vital_group_coverage,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_core_vital_group_coverage.csv
2,cohort_chapter1_site_eligibility,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_site_eligibility.csv
3,cohort_chapter1_site_counts_summary,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_site_counts_summary.csv
4,cohort_chapter1_stay_exclusions,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_stay_exclusions.csv
5,cohort_chapter1_stay_exclusion_summary_by_hospital,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_stay_exclusion_summary_by_hospital.csv
6,cohort_chapter1_counts_by_hospital,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_counts_by_hospital.csv
7,cohort_chapter1_retained_hospitals,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_retained_hospitals.csv
8,cohort_chapter1_retained_stays,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_retained_stays.csv
9,cohort_chapter1_retained_stay_table,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_retained_stay_table.csv


Saved 51 output tables to /Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1
